# Agricultural Price Forecasting

This notebook contains the end-to-end workflow for an agricultural price forecasting project: EDA, feature engineering, time-series modeling, ensemble comparison, model interpretation, and Streamlit deployment data preparation.


*이 프로젝트는 Liam 강사님의 kaggle 머신러닝 강의(메타코드) 학습을 기반으로 구성했습니다. 좋은 강의를 통해 프로젝트 제작과 학습에 도움을 주셔서 감사합니다*

## Project Overview

이 프로젝트는 농산물 유통 데이터를 활용해 주요 품목의 가격을 예측하는 시계열 머신러닝 프로젝트이다.
분석 범위는 데이터 이해, 시계열 특성 검토, 피처 엔지니어링, 모델 비교, 해석 가능성 검토, Streamlit 배포 데이터 준비까지 포함한다.

예측 대상 품목은 다음 7개이다.
- 배추
- 무
- 마늘
- 양파
- 대파
- 건고추
- 깻잎


## Data Loading

데이터는 농산물 가격과 거래량 정보를 포함하며, 본 노트북에서는 가격 예측만을 다룬다.
GitHub 업로드를 고려하여 데이터 경로는 개인 폴더가 아닌 레포지토리 기준 상대경로로 설정한다.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'data').exists():
    ROOT_DIR = ROOT_DIR.parent
DATA_DIR = ROOT_DIR / 'data'

train_file_path = DATA_DIR / 'train.csv'

df = pd.read_csv(train_file_path)
df.head()


In [ ]:
df.shape


### Data Notes

시계열 데이터에서는 정보 누수(leakage)를 방지하는 것이 중요하다.
따라서 피처 설계와 검증 과정에서 미래 시점의 정보가 현재 예측에 사용되지 않도록 주의해야 한다.


In [ ]:
df.columns


In [ ]:
price_columns = [col for col in df.columns if '가격' in col or 'date' in col]
price_columns


In [ ]:
df = df[price_columns]
df.columns = df.columns.str.replace('_가격(원/kg)', '')
prd = ['date', '배추', '무', '마늘', '양파', '대파', '건고추', '깻잎']
df = df[prd]


In [ ]:
df.head()


In [ ]:
test_file_path = DATA_DIR / 'test_2020-11-05.csv'

test = pd.read_csv(test_file_path)
test = test[price_columns]
test.columns = test.columns.str.replace('_가격(원/kg)', '')
test = test[prd]


In [ ]:
test.head()


In [ ]:
import pandas as pd

def load_data_set(file_path, target_prd, kor_to_eng):
    """모델링용 데이터셋을 불러오고 예측 대상 품목만 선택한다."""
    df = pd.read_csv(file_path)
    price_columns = [col for col in df.columns if '가격' in col or 'date' in col]
    df = df[price_columns]
    df.columns = df.columns.str.replace(r'_가격\(원/kg\)', '', regex=True)
    df.columns = df.columns.map(kor_to_eng)
    df['date'] = pd.to_datetime(df['date'])
    return df[target_prd]


In [ ]:
target_prd = ['date', 'cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']
kor_to_eng = {
    'date': 'date',
    '배추': 'cabbage',
    '무': 'radish',
    '마늘': 'garlic',
    '양파': 'onion',
    '대파': 'green_onion',
    '건고추': 'dried_red_pepper',
    '깻잎': 'perilla_leaf',
}

train_file_path = DATA_DIR / 'train.csv'
test_file_path = DATA_DIR / 'test_2020-11-05.csv'


In [ ]:
train_data = load_data_set(train_file_path, target_prd, kor_to_eng)
test_data = load_data_set(test_file_path, target_prd, kor_to_eng)


In [ ]:
train_data.head()


In [ ]:
test_data.head()


In [ ]:
print(train_data.shape)
print(test_data.shape)


In [ ]:
start_date = '2016-01-01'
end_date = '2020-09-28'
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

start_date1 = '2020-09-29'
end_date1 = '2020-11-04'
date_range1 = pd.date_range(start=start_date1, end=end_date1, freq='D')


In [ ]:
date_range


In [ ]:
print(len(date_range) == len(train_data))
print(len(date_range1) == len(test_data))


## EDA

EDA 단계에서는 날짜 누락 여부, 가격 추이, 분포, 계절성, 요일 효과, 품목 간 상관관계를 점검한다.
이 과정은 이후 피처 엔지니어링과 모델 선택의 근거를 제공한다.


In [ ]:
train_data.head()


In [ ]:
print(train_data.info())
print(test_data.info())


In [ ]:
train_data = train_data.sort_values(by='date', ascending=True)

test_data = test_data.sort_values(by='date')


In [ ]:
train_data.head()


In [ ]:
print(train_data.head(1)['date'])
print(train_data.tail(1)['date'])
print("")
print(test_data.head(1)['date'])
print(test_data.tail(1)['date'])


In [ ]:
print(train_data.shape)
print(test_data.shape)


In [ ]:
train_data.drop("date", axis=1).describe().astype(int)


In [ ]:
test_data.drop("date", axis=1).describe().astype(int)


In [ ]:
print(train_data.isnull().sum())
print(test_data.isnull().sum())


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

train_data.set_index("date").plot(figsize=(14,6))

plt.title("The trend of the price columns")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


In [ ]:
def calculate_rolling_mean(data, window=7):
    """
    변동성이 큰 가격을 갖는 품목에 대한 rolling mean 계산

    Args:
        data (pd.DataFrame): DataFrame containing price columns.
        window (int): window size for rolling mean (default is 7 days).

    Returns:
        pd.DataFrame: DataFrame with rolling mean applied to price columns.
    """
    data_rolling_mean = data.drop(columns=['date']).rolling(window=window, min_periods=1).mean()

    data_rolling_mean['date'] = data['date']

    return data_rolling_mean

train_data_roll = calculate_rolling_mean(train_data)


### Rolling Mean

이동평균은 단기 변동성을 완화하고 장기 추세를 확인하기 위한 대표적인 시계열 탐색 기법이다.
본 노트북에서는 7일 이동평균을 사용해 품목별 가격 흐름을 비교한다.


In [ ]:
train_data_roll.head()


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

train_data_roll.set_index("date").plot(figsize=(14,6))

plt.title("The trend of the price columns after rolling mean")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


In [ ]:
kor_to_eng = {
    'date': 'date',
    '배추': 'cabbage',
    '무': 'radish',
    '마늘': 'garlic',
    '양파': 'onion',
    '대파': 'green_onion',
    '건고추': 'dried_red_pepper',
    '깻잎': 'perilla_leaf'
}


### EDA Findings

시각화 결과 품목별 가격 수준과 변동성 패턴이 뚜렷하게 다르게 나타났다.
특히 건고추와 깻잎은 상대적으로 높은 가격대와 큰 변동성을 보였고, 배추와 무는 유사한 흐름을 보이는 구간이 확인되었다.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

sns.histplot(train_data['cabbage'], bins=10, kde=True, label='cabbage', color='blue', edgecolor='white')
sns.histplot(train_data['radish'], bins=10, kde=True, label='radish', color='red', edgecolor='white')
sns.histplot(train_data['garlic'], bins=10, kde=True, label='garlic', color='yellow', edgecolor='white')
sns.histplot(train_data['onion'], bins=10, kde=True, label='onion', color='green', edgecolor='white')

plt.title('Distribution of Prices', fontsize=12)
plt.xlabel('Price', fontsize=10)
plt.ylabel('Density', fontsize=10)
plt.legend()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

sns.histplot(train_data['perilla_leaf'], bins=10, kde=True, label='cabbage', color='blue', edgecolor='white')
sns.histplot(train_data['garlic'], bins=10, kde=True, label='radish', color='red', edgecolor='white')

plt.title('Distribution of Prices', fontsize=12)
plt.xlabel('Price', fontsize=10)
plt.ylabel('Density', fontsize=10)
plt.legend()
plt.show()


In [ ]:
train_data.drop("date", axis=1).describe().astype(int)


In [ ]:
train_data['date'] = pd.to_datetime(train_data['date'])
train_data['day'] = train_data['date'].dt.day_of_week
train_data['month'] = train_data['date'].dt.month

price_by_day = train_data.groupby("day")[['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']].mean().round(2)

price_by_month = train_data.groupby("month")[['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']].mean().round(2)


In [ ]:
price_by_day


In [ ]:
price_by_day.T.plot(kind='bar', figsize=(12, 5))
plt.title('Average Price by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Price')


In [ ]:
price_by_month.T.plot(kind='bar', figsize=(12, 5))
plt.title('Average Price by Month')
plt.xlabel('Month')
plt.ylabel('Average Price')


### Seasonal Patterns

월별 평균 가격을 비교하면 특정 품목은 명절과 계절성의 영향을 크게 받는다.
건고추를 포함한 일부 품목은 특정 월에 가격이 높아지는 경향을 보였다.


In [ ]:
test_data.corr()


In [ ]:
cor_matrix = test_data.corr()[1:].drop("date", axis=1)
cor_matrix.style.background_gradient(cmap='Blues')


### Correlation Patterns

상관관계 분석은 함께 움직이는 품목을 식별하는 데 유용하다.
배추와 무, 무와 대파처럼 상관성이 높은 조합은 공통 수급 요인의 영향을 받을 가능성이 있다.


## Feature Engineering

가격 예측 성능을 높이기 위해 날짜 기반 변수와 시차 기반 변수를 생성한다.
시계열 문제에서는 모델 선택만큼 피처 설계가 성능에 큰 영향을 미친다.


### Feature Engineering Rationale

본 노트북은 날짜 파생 변수, 시차 변수, 이동 통계 변수를 활용해 시계열 패턴을 구조화한다.
이러한 피처는 과거 가격 흐름과 달력 효과를 모델이 동시에 학습할 수 있도록 돕는다.


In [ ]:
train_data = load_data_set(train_file_path, target_prd, kor_to_eng)

test_data = load_data_set(test_file_path, target_prd, kor_to_eng)


In [ ]:
data = pd.concat([train_data, test_data]).reset_index(drop=True)

print(data.head())
print(data.tail())


In [ ]:
import holidays

def add_date_features(data):
    """날짜 기반 파생 변수를 추가한다."""
    data['year'] = data['date'].dt.year
    data['month'] = data['date'].dt.month
    data['day'] = data['date'].dt.day
    data['day_of_week'] = data['date'].dt.dayofweek
    data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)

    kr_holidays = holidays.KR(years=data['year'].unique())
    data['is_holiday'] = data['date'].isin(kr_holidays).astype(int)
    return data


In [ ]:
data = add_date_features(data)


In [ ]:
data


### Temporal Features

날짜에서 연도, 월, 일, 요일, 주말 여부, 공휴일 여부를 추출한다.
이 변수들은 계절성과 휴일 효과를 반영하는 기본 피처로 사용된다.


In [ ]:
def add_lagging_features(data, window=7):
    """
    예측값의 과거 t-1, .. t-n 까지의 데이터를 추가
    Args:
        data (pd.DataFrame): 제품 가격이 포함된 입력 데이터셋.
        window (int, optional): 이전 시간 스텝을 이동할 개수. 기본값은 7.

    Returns:
        pd.DataFrame: 추가된 지연(lag) 피처를 포함한 새로운 데이터셋.
    """
    data = data.copy()
    products = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

    for product in products:
        data[f'{product}_lag_{window}'] = data[product].shift(window)

    return data


In [ ]:
data = add_lagging_features(data, window=7)
data = add_lagging_features(data, window=14)


In [ ]:
data.head()


In [ ]:
def add_rolling_mean(data, window=7):
    """
    변동성이 큰 가격을 갖는 품목에 대한 rolling mean 계산
    Args:
        data (pd.DataFrame): 제품 가격이 포함된 입력 데이터셋.
        window (int, optional): 롤링 윈도우 크기. 기본값은 7.

    Returns:
        pd.DataFrame: rolling mean과 rolling standard deviation을 포함한 새로운 데이터셋.
    """
    products = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

    for product in products:
        data[f'{product}_rolling_mean'] = data[product].rolling(window=window).mean()
        data[f'{product}_rolling_std'] = data[product].rolling(window=window).std()

    return data


In [ ]:
data = add_rolling_mean(data)


### Lag Features

lag 피처는 과거 가격이 미래 가격에 미치는 자기상관 구조를 반영한다.
본 노트북에서는 7일 및 14일 시차 값을 생성하여 단기 패턴과 주간 반복성을 학습한다.


In [ ]:
data.head().T


In [ ]:
def split_data(data, std_date="2020-09-29"):
    """
    특정 날짜 기준으로 train, test dataset 나누기
    Args:
        data (pd.DataFrame): 입력 데이터셋.
        std_date (str, optional): 기준 날짜. 기본값은 '2020-09-29'.

    Returns:
        tuple: 훈련용(train) 데이터셋과 테스트용(test) 데이터셋.
    """
    data = data.dropna()
    train_data = data.query("date < @std_date")
    test_data = data.query("date >= @std_date")

    print("Train set shape:", train_data.shape)
    print("Test set shape:", test_data.shape)
    return train_data, test_data


In [ ]:
train_data, test_data = split_data(data)


In [ ]:
train_data.tail()


In [ ]:
test_data.head()


## Modeling

모델링 단계에서는 시계열 베이스라인과 머신러닝 회귀 모델을 비교한다.
또한 후처리와 앙상블을 통해 예측 성능이 어떻게 달라지는지 함께 확인한다.


### Modeling Strategy

예측 접근은 두 가지 축으로 구성된다.
- 시계열 베이스라인: Prophet
- 지도학습 기반 회귀: Ridge, RandomForest, LightGBM, XGBoost, MLP


### Prophet Baseline

Prophet은 날짜 기반 seasonality와 휴일 효과를 반영할 수 있는 시계열 모델이다.
본 노트북에서는 품목별로 Prophet 모델을 학습한 뒤 테스트 구간 예측값을 생성한다.


In [ ]:
!pip install prophet


In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
logging.getLogger('prophet').setLevel(logging.WARNING)


In [ ]:
train_data = load_data_set(train_file_path, target_prd, kor_to_eng)
test_data = load_data_set(test_file_path, target_prd, kor_to_eng)


In [ ]:
start_dt = '2020-09-29'
end_dt = '2020-11-04'
forecast_df = pd.DataFrame({'date': pd.date_range(start=start_dt, end=end_dt)})

forecast_df


In [ ]:
import holidays

kr_holidays = holidays.KR(years=[2016, 2017, 2018, 2019, 2020])
kr_holidays


In [ ]:
holidays_df = pd.DataFrame(list(kr_holidays.items()), columns=['ds', 'holiday'])
holidays_df['lower_window'] = 0
holidays_df['upper_window'] = 1

holidays_df


In [ ]:
start_dt = '2020-09-29'
end_dt = '2020-11-04'

forecast_df = pd.DataFrame({'date': pd.date_range(start=start_dt, end=end_dt)})
forecast_df.head()


In [ ]:
for col in train_data.columns:
    if col != 'date':
        train_prophet = train_data[['date', col]].rename(columns={'date': 'ds', col: 'y'})

        model_prophet = Prophet(holidays=holidays_df)
        model_prophet.fit(train_prophet)

        future_dates = pd.DataFrame(pd.date_range(start=start_dt, end=end_dt), columns=['ds'])
        forecast = model_prophet.predict(future_dates)
        forecast = forecast.round(2)
        forecast_df[col + '_pred'] = forecast['yhat']


In [ ]:
forecast_df.head()


In [ ]:
forecast_df.tail()


In [ ]:
test = test_data.merge(forecast_df, on=['date'], how='left')
test.head()


In [ ]:
prd = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']
dfs = {}
for p in prd:
    dfs[p] = test[['date', p, p+'_pred']]


In [ ]:
print("Dataframe for galic:")
print(dfs['garlic'].head())


In [ ]:
print("Dataframe for cabbage:")
print(dfs['cabbage'].head())


In [ ]:
num_rows = 3
num_cols = 3
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8))
axes = axes.flatten()

products = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

for i, (prd, ax) in enumerate(zip(products, axes)):
    df = dfs[prd]
    ax.plot(df['date'], df[prd], label='Actual')
    ax.plot(df['date'], df[prd + '_pred'], label='Predicted')
    ax.set_title(prd)
    ax.legend()
    ax.tick_params(axis='x', labelsize=6)

plt.tight_layout()
plt.show()


## Evaluation

모델 평가는 실제값 대비 상대 오차를 기준으로 수행한다.
품목별 가격 수준 차이가 크므로 절대 오차보다 상대 오차 기반 지표가 비교에 적합하다.


In [ ]:
prd = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']
dfs = {}
for p in prd:
    dfs[p] = test[['date', p, p + '_pred']]
    dfs[p]['gap'] = dfs[p][p] - dfs[p][p + '_pred']
    dfs[p]['abs_gap'] = abs(dfs[p][p] - dfs[p][p + '_pred'])
    dfs[p]['ape'] = dfs[p]['abs_gap'] / dfs[p][p]
    dfs[p]['ape_median'] = 1 - dfs[p]['ape'].median()


In [ ]:
for p in prd:
    print(f"{p}: APE Median = {dfs[p]['ape_median'][0]}")


In [ ]:
price_by_day


In [ ]:
test.head()


In [ ]:
test_post = test.copy()

test_post['day'] = test_post['date'].dt.day_of_week

holidays = ['2020-10-01', '2020-10-03']
test_post['holiday'] = test_post['date'].isin(holidays).astype(int)

test_post['is_flag'] = np.where((test_post['day'] == 6) | (test_post['holiday'] == 1), 1, 0)


In [ ]:
test_post


In [ ]:
for p in prd:
    test_post[p + '_pred'] = np.where(test_post['is_flag'] == 0, test_post[p + '_pred'], 0)

test_post.head(7)


In [ ]:
prd = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

dfs_post = {}
for p in prd:
    dfs_post[p] = test_post[['date', p, p + '_pred']]

print("Dataframe for galic:")
print(dfs_post['garlic'].head(10))


In [ ]:
num_rows = 3
num_cols = 3
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8))
axes = axes.flatten()

products = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

for i, (prd, ax) in enumerate(zip(products, axes)):
    df = dfs_post[prd]
    ax.plot(df['date'], df[prd], label='Actual')
    ax.plot(df['date'], df[prd + '_pred'], label='Predicted')
    ax.set_title(prd)
    ax.legend()
    ax.tick_params(axis='x', labelsize=6)

plt.tight_layout()
plt.show()


In [ ]:
prd = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

for p in prd:
    dfs_post[p]['gap'] = (dfs_post[p][p] - dfs_post[p][p + '_pred'])
    dfs_post[p]['abs_gap'] = abs(dfs_post[p][p] - dfs_post[p][p + '_pred'])
    dfs_post[p]['ape'] = dfs_post[p]['abs_gap'] / dfs_post[p][p]
    dfs_post[p]['ape_median'] = 1 - (dfs_post[p]['ape'].median())


In [ ]:
for p in prd:
    print(f"{p}: APE Median = {dfs_post[p]['ape_median'][0]}")


In [ ]:
for p in prd:
    print(f"{p}: APE Median = {dfs[p]['ape_median'][0]}")


### Post-processing Summary

일요일과 공휴일처럼 거래 특성이 뚜렷한 날짜에는 예측값을 규칙 기반으로 보정한다.
이는 도메인 지식을 반영하는 실무형 처리 방식이다.


### Machine Learning Regressors

머신러닝 모델은 날짜 파생 변수와 시차 피처를 함께 사용해 품목별 가격을 예측한다.
단일 모델 성능뿐 아니라 품목별 최적 모델이 달라지는지도 함께 확인한다.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_percentage_error


In [ ]:
train_data.columns


In [ ]:
train_data


In [ ]:
products = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']

features = ['year', 'month', 'day', 'day_of_week', 'is_weekend', 'is_holiday'] + \
           [f'{product}_lag_7' for product in products] + \
           [f'{product}_lag_14' for product in products] + \
           [f'{product}_rolling_mean' for product in products] + \
           [f'{product}_rolling_std' for product in products]

targets = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']


In [ ]:
features


In [ ]:
models = {
    'Ridge': Ridge(),
    'RandomForest': RandomForestRegressor(),
    'LGBM': LGBMRegressor(),
    'XGBoost': XGBRegressor(),
    'MLP': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500)
}

def train_models(train_data, test_data, targets, features, models):
    """
    학습 데이터셋을 받아서 모델별로 반복하여 학습 진행하고 예측값을 반환하는 함수.
    Args:
        train_data (pd.DataFrame): 학습 데이터셋.
        test_data (pd.DataFrame): 테스트 데이터셋.
        targets (list): 예측하려는 품목 리스트.
        features (list): 학습에 사용할 피처 리스트.
        models (dict): 사용할 모델들의 딕셔너리.

    Returns:
        dict: 각 모델에 대해 예측된 값을 포함한 결과 딕셔너리.
    """
    predictions = {model_name: {} for model_name in models}
    for model_name, model in models.items():
        for target in targets:
            model.fit(train_data[features], train_data[target])
            predictions[model_name][target] = model.predict(test_data[features])
    return predictions


In [ ]:
predictions = train_models(train_data, test_data, targets, features, models)


In [ ]:
predictions


In [ ]:
predictions['LGBM']['onion']


In [ ]:
predictions['MLP']['cabbage']


In [ ]:
predictions['XGBoost']['cabbage']


### Model Comparison

예측 결과와 실제값을 함께 비교하여 품목별 성능 차이를 시각적으로 점검한다.
이후 MdAPE 기준 성능표를 통해 모델별 우수 구간을 확인한다.


In [ ]:
def create_predictions_df(test_data, predictions, targets):
    """
    테스트셋에 대한 예측값을 데이터프레임으로 생성하는 함수.

    Args:
        test_data (pd.DataFrame): 테스트 데이터셋.
        predictions (dict): 각 모델에서 예측된 값을 포함한 딕셔너리.
        targets (list): 예측하려는 품목 리스트.

    Returns:
        pd.DataFrame: 테스트 데이터셋에 예측값을 추가한 새로운 데이터프레임.
    """
    test_data_predictions = test_data.copy()

    for model_name, model_preds in predictions.items():
        for target in targets:
            test_data_predictions[f'{target}_pred_{model_name}'] = model_preds[target]
    return test_data_predictions


In [ ]:
test_data_predictions = create_predictions_df(test_data, predictions, targets)


In [ ]:
def evaluate_predictions(test_data_predictions, targets):
    """
    테스트셋에 대한 예측값을 평가하여 MdAPE를 계산하는 함수.

    Args:
        test_data_predictions (pd.DataFrame): 테스트셋과 예측값을 포함한 데이터프레임.
        targets (list): 예측하려는 품목 리스트.

    Returns:
        pd.DataFrame: 각 품목과 모델에 대한 MdAPE 값을 포함하는 데이터프레임.
    """
    MdAPES = {}
    for target in targets:
        for model_name in test_data_predictions.columns:
            if target + '_pred' in model_name:
                APE_col = f'{target}_APE_{model_name.split("_")[2]}'
                test_data_predictions[APE_col] = np.abs(
                    (test_data_predictions[target] - test_data_predictions[model_name]) / test_data_predictions[target]
                )
                MdAPE = test_data_predictions[APE_col].median()
                MdAPES[f'{target}_{model_name.split("_")[2]}'] = 1-round(MdAPE, 6)

    return pd.DataFrame(list(MdAPES.items()), columns=['product_model', 'MdAPE'])


In [ ]:
metrics = evaluate_predictions(test_data_predictions, targets)
metrics[['product', 'model']] = metrics['product_model'].str.split('_', expand=True)


In [ ]:
metrics.head()


In [ ]:
def plot_predictions(test_data_predictions, targets, models):

    num_rows = 3
    num_cols = 3
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 12))
    axes = axes.flatten()

    for i, target in enumerate(targets):
        ax = axes[i]
        ax.plot(test_data_predictions['date'], test_data_predictions[target], label='Actual')
        for model_name in models.keys():
            ax.plot(test_data_predictions['date'], test_data_predictions[f'{target}_pred_{model_name}'],
                    label=f'Predicted_{model_name}')

        ax.set_title(target)
        ax.legend()
        ax.tick_params(axis='x', labelsize=6)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_predictions(test_data_predictions, targets, models)


In [ ]:
def apply_post_processing(test_data, targets):
    """
    post-processing
    args:
        test_data,
        targets
    return:
        test_data
    """
    holidays = ['2020-10-01', '2020-10-03']
    test_data['day'] = test_data['date'].dt.dayofweek
    test_data['holiday'] = test_data['date'].isin(holidays).astype(int)
    test_data['is_flag'] = np.where((test_data['day'] == 6) | (test_data['holiday'] == 1), 1, 0)

    for target in targets:
        for model_name in test_data.columns:
            if target + '_pred_' in model_name:
                test_data[model_name] = np.where(test_data['is_flag'] == 0, test_data[model_name], 0)
    return test_data


In [ ]:
test_data_predictions = create_predictions_df(test_data, predictions, targets)


In [ ]:
test_data_predictions_post = apply_post_processing(test_data_predictions, targets)


In [ ]:
test_data_predictions_post.tail(10)


In [ ]:
metrics_post = evaluate_predictions(test_data_predictions_post, targets)


In [ ]:
metrics_post[['product', 'model']] = metrics_post['product_model'].str.split('_', expand=True)


In [ ]:
metrics_post.head()


In [ ]:
plot_predictions(test_data_predictions_post, targets, models)


In [ ]:
def plot_mdape_compare(metrics_post):
    products = metrics_post['product'].unique()
    num_products = len(products)

    fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 10), sharex=False, sharey=True)

    for i, product in enumerate(products):
        row = i // 3
        col = i % 3
        product_data = metrics_post[metrics_post['product'] == product]

        ax = axes[row, col]

        ax.bar(product_data['model'], product_data['MdAPE'])
        ax.set_title(product)
        ax.set_ylabel('MdAPE')
        ax.tick_params(axis='x', labelrotation=45)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()


In [ ]:
plot_mdape_compare(metrics_post)


In [ ]:
metrics_post.pivot(index='product', columns='model', values='MdAPE').round(5)


In [ ]:
metrics.pivot(index='product', columns='model', values='MdAPE').round(5)


### Ensemble Modeling

성능이 우수한 트리 기반 모델을 중심으로 평균 앙상블과 스태킹을 실험한다.
앙상블은 개별 모델의 분산을 완화하고 더 안정적인 예측을 제공하는지 확인하기 위한 단계이다.


In [ ]:
models = {
    'RandomForest': RandomForestRegressor(),
    'LGBM': LGBMRegressor(),
    'XGBoost': XGBRegressor()
}


In [ ]:
models


In [ ]:
def train_models_avg(train_data, test_data, targets, features, models):
    """
    여러 모델을 학습시키고, 예측값을 평균하여 반환하는 함수
    Args:
        train_data (pd.DataFrame): 학습 데이터셋
        test_data (pd.DataFrame): 테스트 데이터셋
        targets (list of str): 타겟 변수 리스트
        features (list of str): 피처 변수 리스트
        models (dict): 모델의 이름과 객체로 구성된 딕셔너리
    Returns:
        dict: 타겟 변수별 예측값의 평균을 포함하는 딕셔너리
    """
    predictions = {model_name: {} for model_name in models}

    for model_name, model in models.items():
        for target in targets:
            model.fit(train_data[features], train_data[target])
            predictions[model_name][target] = model.predict(test_data[features])

    average_predictions = {}
    for target in targets:
        all_model_preds = np.array([predictions[model_name][target] for model_name in models])
        average_predictions[target] = np.mean(all_model_preds, axis=0)

    return average_predictions


In [ ]:
average_predictions = train_models_avg(train_data, test_data, targets, features, models)


In [ ]:
average_predictions['cabbage'].astype(int)


In [ ]:
def create_predictions_df_avg(test_data, predictions, targets):
    """
    테스트 데이터셋에 대한 예측값을 포함하는 데이터프레임을 생성하는 함수
    Args:
        test_data (pd.DataFrame): 테스트 데이터셋
        predictions (dict): 타겟별 평균 예측값을 담은 딕셔너리
        targets (list of str): 타겟 변수 리스트
    Returns:
        pd.DataFrame: 예측값이 추가된 테스트 데이터프레임
    """
    test_data_predictions = test_data.copy()

    for target in targets:
        test_data_predictions[f'{target}_pred_average'] = predictions[target]

    return test_data_predictions


In [ ]:
test_data_predictions_avg = create_predictions_df_avg(test_data, average_predictions, targets)


In [ ]:
test_data_predictions_avg.head()


In [ ]:
test_data_predictions_post_avg = apply_post_processing(test_data_predictions_avg, targets)


In [ ]:
test_data_predictions_post_avg.head()


In [ ]:
test_data_predictions_post.head()


In [ ]:
metrics_post_avg = evaluate_predictions(test_data_predictions_post_avg, targets)
metrics_post_avg[['product', 'model']] = metrics_post_avg['product_model'].str.split('_', expand=True)


In [ ]:
metrics_post_avg


In [ ]:
metrics_post.head()


In [ ]:
pd.concat([metrics_post, metrics_post_avg])\
  .pivot(index='product', columns='model', values='MdAPE').round(5)


In [ ]:
pd.concat([metrics_post, metrics_post_avg])\
  .pivot(index='product', columns='model', values='MdAPE').round(5)\
  .plot(kind='bar', figsize=(10, 4), rot=0)

plt.ylim(0.6, 1)
plt.tight_layout()


### Stacking Regressor

스태킹은 1단계 모델의 예측을 다시 메타 모델의 입력으로 사용하는 앙상블 방법이다.
본 노트북에서는 RandomForest, LightGBM, XGBoost를 기반 모델로 사용한다.


### Stacking Rationale

평균 앙상블과 달리 스태킹은 모델 간 상호보완적 패턴을 학습할 수 있다.
다만 계산 비용이 높기 때문에 성능 개선 폭과 운영 복잡도를 함께 비교해야 한다.


In [ ]:
from sklearn.ensemble import StackingRegressor


In [ ]:
prediction_stack = {}
base_regressors = [
    ('rf', RandomForestRegressor(random_state=42)),
    ('lgbm', LGBMRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42))
]

stacking_regressor = StackingRegressor(
    estimators=base_regressors,
    final_estimator=XGBRegressor()
)

stacking_regressor


In [ ]:
for target in targets:
    stacking_regressor.fit(train_data[features], train_data[target])
    prediction_stack[target] = stacking_regressor.predict(test_data[features])


In [ ]:
prediction_stack['cabbage'].astype(int)


In [ ]:
def create_predictions_df_stack(test_data, predictions, targets):
    """
    stack 예측용 테스트 데이터셋에 대한 예측값을 포함하는 데이터프레임을 생성하는 함수
    Args:
        test_data (pd.DataFrame): 테스트 데이터셋
        predictions (dict): 타겟별 평균 예측값을 담은 딕셔너리
        targets (list of str): 타겟 변수 리스트
    Returns:
        pd.DataFrame: 예측값이 추가된 테스트 데이터프레임
    """
    test_data_predictions = test_data.copy()

    for target in targets:
        test_data_predictions[f'{target}_pred_stack'] = predictions[target]

    return test_data_predictions


In [ ]:
test_data_predictions_stack = create_predictions_df_stack(test_data, prediction_stack, targets)

test_data_predictions_post_stack = apply_post_processing(test_data_predictions_stack, targets)


In [ ]:
test_data_predictions_post_stack.head()


In [ ]:
metrics_post_stack = evaluate_predictions(test_data_predictions_post_stack, targets)
metrics_post_stack[['product', 'model']] = metrics_post_stack['product_model'].str.split('_', expand=True)
metrics_post_stack


In [ ]:
pd.concat([metrics_post, metrics_post_avg, metrics_post_stack])\
  .pivot(index='product', columns='model', values='MdAPE').round(5)


In [ ]:
pd.concat([metrics_post, metrics_post_avg, metrics_post_stack])\
  .pivot(index='product', columns='model', values='MdAPE').round(5)\
  .plot(kind='bar', figsize=(10,4), rot=0)

plt.ylim(0.6, 1)
plt.tight_layout()


### Deep Learning Experiment

표 형식 시계열 데이터에 대해 간단한 딥러닝 회귀 구조도 비교 대상으로 실험한다.
다만 본 프로젝트의 핵심 비교 대상은 트리 기반 머신러닝과 앙상블 모델이다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np

target_columns = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']
features = [col for col in train_data.columns if col not in ['date'] + target_columns]

scaler = StandardScaler()
train_data[features] = scaler.fit_transform(train_data[features])
test_data[features] = scaler.transform(test_data[features])

class MyDataset(Dataset):
    def __init__(self, data, target_columns, feature_columns):
        self.features = torch.tensor(data[feature_columns].values, dtype=torch.float32)
        self.targets = torch.tensor(data[target_columns].values, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

train_dataset = MyDataset(train_data, target_columns, features)
test_dataset = MyDataset(test_data, target_columns, features)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

class ImprovedFCNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.2):
        super(ImprovedFCNNModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

input_dim = len(features)
hidden_dim = 128
output_dim = len(target_columns)
dropout_rate = 0.3
model = ImprovedFCNNModel(input_dim, hidden_dim, output_dim, dropout_rate)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

early_stopping_patience = 10
best_loss = float('inf')
epochs_no_improve = 0

num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
    avg_loss = total_loss / len(test_loader)

    print(f"Epoch {epoch + 1}/{num_epochs}, Test Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stopping_patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(torch.load('best_model.pt'))


In [ ]:
model.eval()
predictions = {target: [] for target in target_columns}
real_values = {target: [] for target in target_columns}

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        for i, target in enumerate(target_columns):
            predictions[target].extend(outputs[:, i].numpy())
            real_values[target].extend(targets[:, i].numpy())

pred_df = pd.DataFrame(predictions)
real_df = pd.DataFrame(real_values)
date_series = test_data['date'].reset_index(drop=True)

pred_df['date'] = date_series
real_df['date'] = date_series


In [ ]:
def plot_predictions_vs_real(pred_df, real_df, targets):
    num_products = len(targets)
    num_cols = 3
    num_rows = (num_products + num_cols - 1) // num_cols
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(15, 10), sharex=True)
    axes = axes.flatten()

    for i, target in enumerate(targets):
        ax = axes[i]
        ax.plot(real_df['date'], real_df[target], label='Real', color='blue')
        ax.plot(pred_df['date'], pred_df[target], label='Predicted', color='orange')
        ax.set_title(target)
        ax.set_xlabel('Date')
        ax.set_ylabel('Value')
        ax.legend()

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.suptitle("Predictions vs. Real Values")
    plt.show()

plot_predictions_vs_real(pred_df, real_df, target_columns)


### Deep Learning Note

딥러닝은 대규모 비정형 데이터에서 강점을 보이는 경우가 많다.
본 프로젝트와 같은 표 형식 데이터에서는 피처 엔지니어링이 충분히 설계된 트리 기반 모델이 더 효율적일 수 있다.


## Interpretation

모델의 예측 성능뿐 아니라 어떤 피처가 가격 변동에 영향을 주는지도 함께 해석한다.
SHAP을 활용해 주요 변수의 기여 방향과 상대적 중요도를 확인한다.


In [ ]:
import shap


In [ ]:
target = 'perilla_leaf'
model = XGBRegressor().fit(train_data[features], train_data[target])
explainer = shap.TreeExplainer(model, train_data[features])
shap_values = explainer(test_data[features])
shap.summary_plot(shap_values, test_data[features], title=f"SHAP Summary Plot for {target}")


### SHAP Interpretation

SHAP 값은 개별 예측에서 각 피처가 예측값을 얼마나 증가 또는 감소시켰는지 보여준다.
이를 통해 단순 성능 비교를 넘어 모델이 학습한 패턴을 설명할 수 있다.


### Interpretation Summary

시차 변수와 이동 통계 변수는 가격 예측에서 핵심 피처로 작동한다.
해석 결과는 시계열 특성을 반영한 피처 설계가 모델 성능 향상에 중요하다는 점을 보여준다.


## Streamlit Deployment

학습 및 평가가 끝난 뒤에는 예측 결과를 Streamlit에서 시각화할 수 있도록 별도의 데이터셋으로 정리한다.
이 단계는 분석 결과를 웹 애플리케이션 형태로 전달하기 위한 준비 과정이다.


### Dashboard Scope

대시보드는 품목 선택, 날짜 구간 확인, 모델별 예측 결과 비교 기능을 제공하도록 구성한다.
실제값과 예측값을 함께 시각화해 비개발자도 결과를 빠르게 이해할 수 있도록 설계한다.


In [ ]:
data.head()


In [ ]:
targets = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']


In [ ]:
real = data.set_index('date')[targets].reset_index()
real


In [ ]:
test_data_predictions_post.head()


In [ ]:
pred_cols = [col for col in test_data_predictions_post.columns if 'pred' in col or 'date' in col]
pred_cols_avg = [col for col in test_data_predictions_post_avg.columns if 'pred' in col or 'date' in col]
pred_cols_stack = [col for col in test_data_predictions_post_stack.columns if 'pred' in col or 'date' in col]


In [ ]:
pred_cols_avg


In [ ]:
pred = test_data_predictions_post[pred_cols]
pred_avg = test_data_predictions_post_avg[pred_cols_avg]
pred_stack = test_data_predictions_post_stack[pred_cols_stack]


In [ ]:
pred_avg.head()


In [ ]:
real = real.merge(pred, on='date', how='left')\
           .merge(pred_avg, on='date', how='left')\
           .merge(pred_stack, on='date', how='left')


In [ ]:
real.tail().T


In [ ]:
real.columns


In [ ]:
real.shape


In [ ]:
real[['date', 'cabbage', 'cabbage_pred_LGBM', 'cabbage_pred_XGBoost', 'cabbage_pred_average', 'cabbage_pred_stack']][-100:].set_index("date").plot(figsize=(15, 5))


In [ ]:
file_path = DATA_DIR / 'streamlit_data.csv'
real.to_csv(file_path, index=False)


In [ ]:
metric_summay = pd.concat([metrics_post, metrics_post_avg, metrics_post_stack])\
  .pivot(index='product', columns='model', values='MdAPE').round(5).reset_index()


In [ ]:
metric_summay


In [ ]:
file_path = DATA_DIR / 'metric_summary.csv'
metric_summay.to_csv(file_path, index=False)


### Streamlit App

다음 코드는 Streamlit 애플리케이션에서 사용할 예시 로직이다.
저장된 예측 데이터와 성능 요약 데이터를 불러와 품목별 시각화를 제공한다.


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_file_path = 'data/streamlit_data.csv'

@st.cache_data
def load_data(file_path):
    return pd.read_csv(file_path)

df = load_data(csv_file_path)

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
else:
    st.error("Date column not found in the CSV file.")

def preprocess_data(df):
    cutoff_date = pd.to_datetime('2020-09-28')
    cols_to_zero = ['cabbage', 'radish', 'garlic', 'onion', 'green_onion', 'dried_red_pepper', 'perilla_leaf']
    df.loc[df.index > cutoff_date, cols_to_zero] = np.nan
    return df

def plot_predictions_over_time(df, vegetables, rolling_mean_window):
    fig, ax = plt.subplots(figsize=(14, 7))

    colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
    num_colors = len(colors)

    for i, veg in enumerate(vegetables):
        ax.plot(df.index, df[veg], label=veg, linewidth=2, color=colors[i % num_colors])
        rolling_mean = df[veg].rolling(window=rolling_mean_window).mean()
        ax.plot(df.index, rolling_mean, label=f'{veg} ({rolling_mean_window}-day Rolling Mean)', linestyle='--', color=colors[i % num_colors])

    ax.set_xlabel('Date', fontsize=14)
    ax.set_ylabel('Price', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(True, color='lightgrey', linestyle='--')
    fig.tight_layout()
    st.pyplot(fig)

df = preprocess_data(df)

metric_file_path = 'data/metric_summary.csv'

metric_summary = pd.read_csv(metric_file_path)
metric_summary.set_index('product', inplace=True)

st.title('🍇농산물 가격 예측 대시보드🥭')
st.markdown("""
    왼쪽에서 품목과 예측모델, 날짜를 입력하면 특정기간 이후 예측 가격이 표시됩니다.
    """)

st.sidebar.title('조회 기간')
start_date = st.sidebar.date_input('시작일', df.index.min())
end_date = st.sidebar.date_input('마지막일', df.index.max())

st.sidebar.title('품목을 선택해주세요')
sorted_vegetables = sorted(df.columns)
vegetables = st.sidebar.multiselect('조회 품목:', sorted_vegetables)
rolling_mean_window = st.sidebar.slider('Rolling Mean Window', min_value=1, max_value=30, value=7)

st.sidebar.markdown("""
| Korean | English    |
|--------|------------|
| 배추   | cabbage    |
| 무     | radish     |
| 마늘   | garlic     |
| 양파   | onion      |
| 대파   | green_onion     |
| 건고추 | dried_red_pepper   |
| 깻잎   | perilla_leaf  |
""")

filtered_df = df.loc[start_date:end_date]

if vegetables:
    st.subheader('품목별 예측 대시보드')
    plot_predictions_over_time(filtered_df, vegetables, rolling_mean_window)

if st.checkbox('Show Filtered DataFrame'):
    st.write(filtered_df)

st.subheader('정확도 Summary')
st.write(metric_summary)


### Deployment Summary

본 노트북은 모델 학습과 평가, 그리고 배포용 데이터 생성까지의 전 과정을 정리한다.
실제 서비스 레포지토리에서는 `app.py`와 `data/` 디렉터리를 이용해 Streamlit 앱을 실행할 수 있다.


### Additional Notes

향후 개선 방향으로는 외부 변수 추가, 품목별 전용 모델 운영, 시계열 교차검증 고도화, 자동 배치 예측 파이프라인 구축 등을 고려할 수 있다.
